# 04. Pelatihan, Efisiensi Komputasi, dan Evaluasi 5 Model Klasifikasi

Notebook ini menjalankan **Tahap 4 metodologi penelitian** — melatih dan mengevaluasi secara penuh **5 algoritma pembelajaran mesin** (*Support Vector Machine*, *Logistic Regression*, *Naïve Bayes*, *Random Forest*, dan *Gradient Boosting*) dari sumber tunggal `data/basic_articles_dataset.csv` untuk kedua pendekatan:
1. **Dengan Semantik *Frame*** (`ngram_range=(2, 3)`, retensi *stopwords*, split `85% / 10% / 5%`).
2. **Medan Makna / Tanpa Semantik *Frame*** (`ngram_range=(1, 1)`, penghapusan *stopwords*, split `80% / 15% / 5%`).

> **Tip Performa (RAM 8 GB / Mac M1 Air):** Untuk menghindari overhead memori kernel Jupyter dan mendukung fitur *auto-resume checkpoint*, Anda dapat menjalankan skrip terminal hemat RAM:
> ```bash
> .venv/bin/python scripts/train_all_models.py --models SVC LogisticRegression MultinomialNB
> .venv/bin/python scripts/train_all_models.py
> ```
> Setelah skrip selesai (atau langsung dari nol jika `SKIP_EXISTING = False`), jalankan sel-sel di bawah ini untuk menampilkan tabel perbandingan, grafik, *Confusion Matrix*, dan verifikasi pemakaian ulang kelima model `.joblib`.


In [ ]:
import sys
from pathlib import Path

# Resolve root repositori secara relatif
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "src" / "arabic_news").exists() else cwd.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT.name}")

import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from arabic_news.config import (
    FIGURES_DIR,
    LABELS,
    LABELS_ID,
    METRICS_DIR,
    MODELS_DIR,
    RANDOM_SEED,
    STATS_DIR,
    resolve_data_file,
)
from arabic_news.evaluation import (
    evaluate_model,
    parse_evaluation_metrics_txt,
    parse_resource_log_txt,
    prepare_medan_and_frame_splits,
    save_evaluation_metrics_txt,
    save_resource_log_txt,
    train_classifier_with_monitoring,
)
from arabic_news.features import create_tfidf_vectorizer
from arabic_news.models import (
    MODEL_FILE_SLUGS,
    build_deployable_svm_bundle,
    predict_with_reusable_model,
    save_deployable_svm_bundle,
    save_reusable_model,
    save_vectorizer,
)
from arabic_news.preprocessing import reshape_arabic_for_plot

plt.rcParams["figure.dpi"] = 120
SKIP_EXISTING = True


## 4.1 Pemeriksaan *Checkpoint*, Split, & Vektorisasi TF-IDF dari `basic_articles_dataset.csv`


In [ ]:
MODELS_ORDER = [
    ("SVC", "SVM (LinearSVC C=10)", "svm.txt", "SVM"),
    ("LogisticRegression", "Logistic Regression", "logistic.txt", "LogReg"),
    ("MultinomialNB", "Naïve Bayes", "naive.txt", "NaiveBayes"),
    ("RandomForest", "Random Forest", "forest.txt", "RandomForest"),
    ("GradientBoosting", "Gradient Boosting", "gradient.txt", "GradBoost"),
]

def _has_checkpoint(key, res_file, scheme):
    subdir = "frame_semantik" if scheme == "frame_semantik" else "medan_semantik"
    slug = MODEL_FILE_SLUGS[key]
    req = [
        METRICS_DIR / subdir / f"no_lemma_evaluation_metrics_{key}.txt",
        METRICS_DIR / "cpu_and_memory" / subdir / res_file,
        MODELS_DIR / "all_models" / f"{slug}_{scheme}.joblib",
        MODELS_DIR / "all_models" / f"vectorizer_{scheme}.joblib",
    ]
    if key == "SVC":
        req.append(MODELS_DIR / f"svm_{scheme}.joblib")
    return all(p.exists() for p in req)

need_training = [
    (k, d, r, s) for k, d, r, s in MODELS_ORDER
    if (not SKIP_EXISTING)
    or (not _has_checkpoint(k, r, "frame_semantik"))
    or (not _has_checkpoint(k, r, "medan_makna"))
]

if need_training:
    print(f"Menyiapkan data & TF-IDF untuk model: {[m[0] for m in need_training]}")
    basic_csv = resolve_data_file("basic_articles_dataset.csv")
    df_basic = pd.read_csv(basic_csv, usecols=["label", "text"], low_memory=False)
    splits = prepare_medan_and_frame_splits(df_basic, random_state=RANDOM_SEED)
    del df_basic
    gc.collect()

    tr_f_df = splits["frame_semantik"]["train_df"]
    val_f_df = splits["frame_semantik"]["val_df"]
    vec_frame = create_tfidf_vectorizer(ngram_range=(2, 3))
    X_tr_frame = vec_frame.fit_transform(tr_f_df["text"])
    X_val_frame = vec_frame.transform(val_f_df["text"])
    y_tr_frame = tr_f_df["label"]
    y_val_frame = val_f_df["label"]
    save_vectorizer(vec_frame, "frame_semantik", compress=1)
    print(f"TF-IDF Semantik Frame (2,3) : train={X_tr_frame.shape}, val={X_val_frame.shape}")

    tr_m_txt = splits["medan_makna"]["train_text"]
    val_m_txt = splits["medan_makna"]["val_text"]
    y_tr_medan = splits["medan_makna"]["train_labels"]
    y_val_medan = splits["medan_makna"]["val_labels"]
    vec_medan = create_tfidf_vectorizer(ngram_range=(1, 1))
    X_tr_medan = vec_medan.fit_transform(tr_m_txt)
    X_val_medan = vec_medan.transform(val_m_txt)
    save_vectorizer(vec_medan, "medan_makna", compress=1)
    print(f"TF-IDF Medan Makna (1,1)    : train={X_tr_medan.shape}, val={X_val_medan.shape}")
else:
    print("Seluruh checkpoint metrik & model .joblib sudah tersedia; melewati pelatihan ulang.")


## 4.2 Pelatihan, Pemantauan CPU/RAM, Penyimpanan 5 Model `.joblib`, & Evaluasi Lengkap

- **Kelima model untuk kedua pendekatan** disimpan ke `models/all_models/<model>_<skema>.joblib` agar setiap algoritma dapat dipakai ulang kapan saja melalui `load_reusable_model()` atau `predict_with_reusable_model()`.
- **Model terbaik SVM (`LinearSVC C=10`)** juga dikemas sebagai bundel kompak (`<50 MB`) di `models/svm_frame_semantik.joblib` dan `models/svm_medan_makna.joblib` untuk dipakai oleh aplikasi Streamlit (`app/streamlit_app.py`) dan Notebook 05.


In [ ]:
for key, display_name, res_file, _ in need_training:
    if (not SKIP_EXISTING) or (not _has_checkpoint(key, res_file, "frame_semantik")):
        print(f"Melatih {display_name} (Semantik Frame 2,3)...")
        clf_f, res_f = train_classifier_with_monitoring(
            key, X_tr_frame, y_tr_frame, mode="frame", random_state=RANDOM_SEED
        )
        ev_f = evaluate_model(clf_f, X_val_frame, y_val_frame)
        save_evaluation_metrics_txt(ev_f, key, METRICS_DIR / "frame_semantik" / f"no_lemma_evaluation_metrics_{key}.txt")
        save_resource_log_txt(res_f, key, METRICS_DIR / "cpu_and_memory" / "frame_semantik" / res_file)
        save_reusable_model(
            clf_f, model_key=key, scheme="frame_semantik", ngram_range=(2, 3), accuracy=ev_f["accuracy"], compress=1
        )
        if key == "SVC":
            bundle_frame = build_deployable_svm_bundle(
                vec_frame, clf_f, mode="frame_semantik", ngram_range=(2, 3),
                max_features=700_000, full_accuracy=ev_f["accuracy"],
            )
            ev_f_compact = evaluate_model(
                bundle_frame["classifier"], bundle_frame["vectorizer"].transform(val_f_df["text"]), y_val_frame
            )
            bundle_frame["compact_accuracy"] = ev_f_compact["accuracy"]
            save_deployable_svm_bundle(bundle_frame, MODELS_DIR / "svm_frame_semantik.joblib")

    if (not SKIP_EXISTING) or (not _has_checkpoint(key, res_file, "medan_makna")):
        print(f"Melatih {display_name} (Medan Makna 1,1)...")
        clf_m, res_m = train_classifier_with_monitoring(
            key, X_tr_medan, y_tr_medan, mode="frame", random_state=RANDOM_SEED
        )
        ev_m = evaluate_model(clf_m, X_val_medan, y_val_medan)
        save_evaluation_metrics_txt(ev_m, key, METRICS_DIR / "medan_semantik" / f"no_lemma_evaluation_metrics_{key}.txt")
        save_resource_log_txt(res_m, key, METRICS_DIR / "cpu_and_memory" / "medan_semantik" / res_file)
        save_reusable_model(
            clf_m, model_key=key, scheme="medan_makna", ngram_range=(1, 1), accuracy=ev_m["accuracy"], compress=1
        )
        if key == "SVC":
            bundle_medan = build_deployable_svm_bundle(
                vec_medan, clf_m, mode="medan_makna", ngram_range=(1, 1),
                max_features=500_000, full_accuracy=ev_m["accuracy"],
            )
            ev_m_compact = evaluate_model(
                bundle_medan["classifier"], bundle_medan["vectorizer"].transform(val_m_txt), y_val_medan
            )
            bundle_medan["compact_accuracy"] = ev_m_compact["accuracy"]
            save_deployable_svm_bundle(bundle_medan, MODELS_DIR / "svm_medan_makna.joblib")

eff_rows, perf_rows, active_shorts = [], [], []
svc_eval_by_scheme = {}

for key, display_name, res_file, short_name in MODELS_ORDER:
    mf_path = METRICS_DIR / "frame_semantik" / f"no_lemma_evaluation_metrics_{key}.txt"
    mn_path = METRICS_DIR / "medan_semantik" / f"no_lemma_evaluation_metrics_{key}.txt"
    if not mf_path.exists() or not mn_path.exists():
        continue

    mf = parse_evaluation_metrics_txt(mf_path)
    mn = parse_evaluation_metrics_txt(mn_path)
    rf = parse_resource_log_txt(METRICS_DIR / "cpu_and_memory" / "frame_semantik" / res_file)
    rn = parse_resource_log_txt(METRICS_DIR / "cpu_and_memory" / "medan_semantik" / res_file)

    if key == "SVC":
        svc_eval_by_scheme["frame"] = mf
        svc_eval_by_scheme["medan"] = mn

    mem_inc = (
        round((rf["max_memory_mb"] - rn["max_memory_mb"]) / rn["max_memory_mb"] * 100, 2)
        if rn["max_memory_mb"]
        else 0.0
    )
    eff_rows.append({
        "Model": display_name,
        "Waktu Frame (s)": round(mf["execution_time_seconds"], 4),
        "Waktu Medan (s)": round(mn["execution_time_seconds"], 4),
        "RAM Frame (MB)": round(rf["max_memory_mb"], 2),
        "RAM Medan (MB)": round(rn["max_memory_mb"], 2),
        "Kenaikan RAM (%)": mem_inc,
        "CPU Frame (%)": round(rf["avg_cpu_percent"], 2),
        "CPU Medan (%)": round(rn["avg_cpu_percent"], 2),
    })
    perf_rows.append({
        "Model": display_name,
        "Acc Frame (2,3)": round(mf["accuracy"], 4),
        "Prec Frame": round(mf["precision"], 4),
        "Rec Frame": round(mf["recall"], 4),
        "F1 Frame": round(mf["f1_score"], 4),
        "Support Frame": int(mf["support"]),
        "Acc Medan (1,1)": round(mn["accuracy"], 4),
        "Prec Medan": round(mn["precision"], 4),
        "Rec Medan": round(mn["recall"], 4),
        "F1 Medan": round(mn["f1_score"], 4),
        "Support Medan": int(mn["support"]),
    })
    active_shorts.append(short_name)

eff_df = pd.DataFrame(eff_rows)
perf_df = pd.DataFrame(perf_rows)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
eff_df.to_csv(METRICS_DIR / "efficiency_comparison.csv", index=False)
perf_df.to_csv(METRICS_DIR / "effectiveness_comparison.csv", index=False)

display(eff_df)
display(perf_df)


## 4.3 Performa Per-Kategori, Grafik Komparasi & *Confusion Matrix* Model Terbaik SVM


In [ ]:
ev_svc_f = svc_eval_by_scheme["frame"]
per_class_map = ev_svc_f["per_class"]

per_class_df = pd.DataFrame([
    {
        "Label": cid,
        "Kategori": f"{LABELS[cid]} ({LABELS_ID[cid]})",
        "Precision": round(per_class_map[cid]["precision"], 4),
        "Recall": round(per_class_map[cid]["recall"], 4),
        "F1-Score": round(per_class_map[cid]["f1-score"], 4),
        "Support": int(per_class_map[cid]["support"]),
    }
    for cid in range(1, 12)
])
display(per_class_df)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
x = np.arange(len(perf_df))
w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
min_acc = float(min(perf_df["Acc Frame (2,3)"].min(), perf_df["Acc Medan (1,1)"].min()) * 100)
axes[0].bar(x - w/2, perf_df["Acc Frame (2,3)"] * 100, w, label="Dengan Frame (2,3)")
axes[0].bar(x + w/2, perf_df["Acc Medan (1,1)"] * 100, w, label="Tanpa Frame / Medan (1,1)")
axes[0].set_title("Perbandingan Akurasi (%)")
axes[0].set_ylim(max(0.0, min_acc - 5.0), 100.0)
axes[0].set_xticks(x, active_shorts, rotation=20)
axes[0].legend()

axes[1].bar(x - w/2, eff_df["Waktu Frame (s)"], w, label="Dengan Frame (2,3)")
axes[1].bar(x + w/2, eff_df["Waktu Medan (s)"], w, label="Tanpa Frame / Medan (1,1)")
axes[1].set_title("Waktu Evaluasi (detik)")
axes[1].set_xticks(x, active_shorts, rotation=20)
axes[1].legend()

axes[2].bar(x - w/2, eff_df["RAM Frame (MB)"], w, label="Dengan Frame (2,3)")
axes[2].bar(x + w/2, eff_df["RAM Medan (MB)"], w, label="Tanpa Frame / Medan (1,1)")
axes[2].set_title("Konsumsi RAM Puncak (MB)")
axes[2].set_xticks(x, active_shorts, rotation=20)
axes[2].legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison.png", dpi=150)
plt.show()

labels_ar = [reshape_arabic_for_plot(LABELS[i]) for i in range(1, 12)]
fig_cm = plt.figure(figsize=(10, 8.5))
sns.heatmap(
    ev_svc_f["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels_ar,
    yticklabels=labels_ar,
)
plt.title(f"Confusion Matrix SVM Dengan Semantik Frame (Akurasi: {ev_svc_f['accuracy']*100:.2f}%)")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
fig_cm.savefig(FIGURES_DIR / "confusion_matrix_svm_frame.png", dpi=150)
plt.show()


## 4.4 Verifikasi & Contoh Pemakaian Ulang (*Reuse*) Kelima Model `.joblib`

Setiap model di `models/all_models/` yang sudah selesai dilatih dapat dimuat ulang tanpa pelatihan ulang melalui `load_reusable_model()` atau `predict_with_reusable_model()`. Sementara itu, aplikasi interaktif Streamlit (`app/streamlit_app.py`) khusus memuat bundel **model terbaik SVM (`models/svm_frame_semantik.joblib` & `models/svm_medan_makna.joblib`)**.


In [ ]:
sample_csv = STATS_DIR / "sample_11_test_articles.csv"
sample_probe = (
    str(pd.read_csv(sample_csv, nrows=1).iloc[0]["text"])
    if sample_csv.exists()
    else "فوز المنتخب الوطني في بطولة كأس العالم في المباراة النهائية"
)

reuse_rows = []
for key, display_name, res_file, _ in MODELS_ORDER:
    if _has_checkpoint(key, res_file, "frame_semantik") and _has_checkpoint(key, res_file, "medan_makna"):
        pred_f = predict_with_reusable_model(sample_probe, model_key=key, scheme="frame_semantik")
        pred_m = predict_with_reusable_model(sample_probe, model_key=key, scheme="medan_makna")
        reuse_rows.append({
            "Model (.joblib)": display_name,
            "Prediksi Frame (2,3)": f"{pred_f['predicted_category']} ({pred_f['predicted_category_id']})",
            "Prediksi Medan (1,1)": f"{pred_m['predicted_category']} ({pred_m['predicted_category_id']})",
        })

display(pd.DataFrame(reuse_rows))
print("Daftar seluruh berkas model .joblib di models/:")
for p in sorted(MODELS_DIR.rglob("*.joblib")):
    print(f" - {p.relative_to(REPO_ROOT)} ({p.stat().st_size / (1024 * 1024):.2f} MB)")
